# Fase 1 — Entendimento do Negócio

**Projeto:** Previsão de demanda e política de estoque — Rede Bandeirante
**Metodologia:** CRISP-DM (Cross Industry Standard Process for Data Mining)
**Natureza deste notebook:** modelagem conceitual do negócio (praticamente só markdown; nenhum dado é carregado).

## Objetivo da fase
Traduzir um problema de negócio em um problema analítico. O caso começa com a pergunta de um comprador de um
centro de distribuição (CD) que abastece 144 lojas: *quanto pedir* de cada item, a cada ciclo de reposição.
Esta fase fixa o horizonte de previsão, explicita a assimetria econômica do erro e define os critérios pelos
quais o trabalho será julgado pelo comitê de investimentos.

## Achados e decisões registrados nesta fase
- O projeto **não é de previsão**, é de **decisão de compra**: a previsão é apenas um insumo de uma política de pedido.
- **Horizonte de previsão H = 28 dias** (lead time de 14 dias + ciclo de revisão de 14 dias).
- **Razão de custo faltar:sobrar = 15:1** (faltar custa 30% do preço; sobrar custa 2% do preço no período).
- **Nível de serviço ótimo = 30/(30+2) = 93,75%** — deliberadamente menor que 100%.
- **Métricas de avaliação:** WMAPE (erro absoluto percentual ponderado pelo volume) e **viés / MPE**.
- **Baseline oficial do processo atual:** média das últimas 4 semanas de venda.
- **Três critérios de sucesso**, sendo o **custo total em reais o decisivo** — falhar nele mata o projeto.
- Restrições do ciclo: lead time e ciclo de revisão são contratuais; histórico de pouco menos de três anos;
  decisões comerciais não estão registradas nas bases.


## 1. O caso e o contexto de negócio

A **Rede Bandeirante** é uma rede supermercadista regional de 144 lojas no interior de São Paulo e sul de
Minas Gerais, abastecida por um **único centro de distribuição em Jundiaí**. O crescimento se deu por
adensamento geográfico em torno do CD — vantagem logística que concentra todo o risco de abastecimento em
um único elo: se o CD erra a compra, as 144 lojas erram juntas.

O varejo alimentar opera com margem líquida estreita (1% a 3%). O resultado do exercício é definido por
duas falhas que puxam em direções opostas:

- **Ruptura (faltar):** destrói uma venda já disponível para acontecer. Em mercearia seca, o consumidor
  raramente adia — substitui pela marca ao lado ou compra no concorrente da esquina. É perda irreversível.
- **Excesso (sobrar):** imobiliza capital de giro e gera custo de carrego/remarcação, mas é reversível —
  o produto continua no ativo e tende a ser vendido.

Quem decide é o comprador do CD. Hoje o processo cabe em uma frase: **a média das quatro últimas semanas de
venda, ajustada no olho quando o comprador lembra de alguma promoção**. Ao mesmo tempo, a empresa convive com
94% de ocupação de armazém **e** ruptura de fim de semana — sinal de que o problema não é o volume total,
mas comprar o volume certo do produto errado, no momento errado.

Quatro áreas consomem a sugestão de pedido, com incentivos distintos: o comprador (nível de serviço);
a diretoria de supply chain (ocupação e doca); a diretoria comercial (calendário promocional, fechado com
6 a 8 semanas de antecedência); e a diretoria financeira (capital de giro imobilizado).


## 2. As seis fases do CRISP-DM

O projeto segue o CRISP-DM, com uma leitura própria dos dados de varejo. As fases e o que cada uma
entrega neste caso:

1. **Entendimento do Negócio (este notebook):** qual decisão está sendo tomada, quais parâmetros são
   contratuais, qual a economia do erro e quais os critérios de sucesso.
2. **Entendimento dos Dados:** o que cada coluna de fato mede, o que ela deixa de medir, e quais registros
   não deveriam estar no escopo de um modelo de previsão.
3. **Preparação dos Dados:** grão de trabalho, remoção de itens fora de escopo, definição do horizonte H e
   separação temporal estrita entre treino e teste.
4. **Modelagem:** baselines (média móvel, naive sazonal) antes de qualquer sofisticação; depois modelos
   com sazonalidade (Holt-Winters, SARIMA) e com variável exógena (SARIMAX + promoção).
5. **Avaliação:** WMAPE, viés e — sobretudo — simulação de custo total com ruptura e excesso, com backtest
   em múltiplas origens para que o número prometido ao comitê se sustente ao longo do tempo.
6. **Implantação:** tradução da previsão em uma **política de pedido** (nível de serviço e estoque de
   segurança) e no que aparece na tela do comprador na quinta-feira de manhã.


## 3. Dedução do horizonte de previsão (H = 28 dias)

Dois parâmetros são contratuais e não estão sob controle da equipe:

- **Lead time (L):** 14 dias entre o pedido e o recebimento na doca.
- **Ciclo de revisão (R):** só é possível fazer um novo pedido ao mesmo fornecedor 14 dias depois.

Um pedido feito hoje precisa cobrir todo o intervalo até que o **próximo** pedido chegue — ou seja,
o lead time **mais** o ciclo de revisão. Esse é o horizonte que a previsão precisa atender.


In [1]:
# Parâmetros operacionais contratuais (Exhibit 1 do caso)
lead_time_dias = 14          # do pedido até o recebimento na doca
ciclo_revisao_dias = 14      # intervalo mínimo entre dois pedidos ao mesmo fornecedor

# O pedido feito hoje precisa cobrir todo o intervalo até a chegada do PRÓXIMO pedido.
H = lead_time_dias + ciclo_revisao_dias
print(f"Horizonte de previsão H = lead time ({lead_time_dias} dias) + ciclo de revisão ({ciclo_revisao_dias} dias) = {H} dias")


Horizonte de previsão H = lead time (14 dias) + ciclo de revisão (14 dias) = 28 dias


## 4. A assimetria econômica do erro: razão de custo 15:1

Os dois erros não custam a mesma coisa:

- **Faltar 1 unidade:** ≈ **30% do preço** (margem que evapora; a venda migra para o concorrente).
- **Sobrar 1 unidade no período:** ≈ **2% do preço** (capital e armazenagem, reversível).

Logo, **errar para baixo é 15 vezes mais caro do que errar para cima**. Decidir quanto pedir não é apenas
acertar a previsão: é escolher deliberadamente para que lado errar.


In [2]:
# Parâmetros econômicos do Exhibit 1 (dados pelo caso)
custo_faltar = 0.30   # 30% do preço -> margem perdida por unidade faltante
custo_sobrar = 0.02   # 2%  do preço -> carrego por unidade excedente no período

razao_custo = custo_faltar / custo_sobrar
print(f"Custo de faltar (por unidade): {custo_faltar:.0%} do preço")
print(f"Custo de sobrar (por unidade): {custo_sobrar:.0%} do preço")
print(f"Razão de custo faltar:sobrar = {custo_faltar}/{custo_sobrar} = {razao_custo:.0f}:1")


Custo de faltar (por unidade): 30% do preço
Custo de sobrar (por unidade): 2% do preço
Razão de custo faltar:sobrar = 0.3/0.02 = 15:1


## 5. Nível de serviço ótimo (newsvendor) — 93,75%

O nível de serviço ótimo é a razão de **sobrecusto por faltar** sobre o **sobrecusto total**:

> Nível de serviço = C_faltar / (C_faltar + C_sobrar) = 0,30 / (0,30 + 0,02) = **0,9375 = 93,75%**

O alvo **não é 100%**. Perseguir 100% significaria carregar estoque para cobrir até um cenário improvável
de demanda alta, pagando carrego (2% por unidade) para evitar uma perda (30% por unidade) que, na cauda,
é cada vez mais rara. Como faltar é 15× mais custoso, a política correta se posiciona **acima da média**
da distribuição de demanda — exatamente 93,75% da probabilidade acumulada — deixando ~6,25% da cauda
superior como risco deliberado, e a gôndola cheia na maior parte do tempo.


In [3]:
nivel_servico_otimo = custo_faltar / (custo_faltar + custo_sobrar)
print(f"Nível de serviço ótimo = {custo_faltar} / ({custo_faltar} + {custo_sobrar}) = "
      f"{nivel_servico_otimo:.4f}  ({nivel_servico_otimo*100:.2f}%)")
print()
print("Leitura: pedir para atender 93,75% da probabilidade de demanda; a folga (estoque de segurança)")
print("cobre o restante do risco. Perseguir 100% seria carregar estoque em excesso.")


Nível de serviço ótimo = 0.3 / (0.3 + 0.02) = 0.9375  (93.75%)

Leitura: pedir para atender 93,75% da probabilidade de demanda; a folga (estoque de segurança)
cobre o restante do risco. Perseguir 100% seria carregar estoque em excesso.


## 6. Métricas de avaliação: WMAPE e viés (MPE)

Precisamos de métricas que conversem com a economia do problema e que sejam robustas a itens de baixo volume.

- **WMAPE (Weighted Mean Absolute Percentage Error)** — erro percentual absoluto ponderado pelo volume.
  Mede a **magnitude** do erro sem deixar que itens pequenos distorçam a média:

  `WMAPE = soma(|real − previsto|) / soma(real) × 100`

- **Viés (MPE — Mean Percentage Error)** — mede a **direção** do erro, não a magnitude.
  O sinal importa: `viés = soma(real − previsto) / soma(real) × 100`.

Interpretação: um modelo com WMAPE baixo mas viés sistematicamente negativo está **subestimando** a demanda
(risco de ruptura); viés positivo está **superestimando** (risco de excesso). O segundo critério exige viés
próximo de zero — sem tendência estrutural de faltar ou sobrar.

O **baseline oficial** é a **média das últimas 4 semanas** de venda (o processo vigente do comprador).
Todo modelo precisa vencê-lo antes de se discutir sofisticação.


## 7. Critérios de sucesso (em ordem — o último é o que manda)

Fixados por escrito pela diretoria de supply chain, **com pesos diferentes**:

1. **Superar o processo atual** (média das últimas 4 semanas) em erro de previsão (WMAPE).
2. **Apresentar viés próximo de zero**, sem tendência estrutural de faltar ou de sobrar.
3. **Reduzir o custo total em reais** contra o processo atual, simulando explicitamente ruptura e excesso
   com os custos unitários do Exhibit 1. **Este critério é decisivo:** falhar nele mata o projeto, mesmo que
   os dois primeiros sejam cumpridos com folga.

## Conexão com as próximas fases
A Fase 1 deixa três perguntas operacionais em aberto, respondidas nas fases seguintes:
- **O que as colunas de fato medem?** (Fase 2) — a coluna `unidades` é **venda**, não demanda; e a coluna de
  estoque é saldo de fim de dia.
- **Em que nível prever e como separar treino/teste sem contaminar o futuro?** (Fase 3) — grão produto × dia,
  H = 28 dias e corte temporal estrito, sem embaralhar.
- **Como transformar a previsão em decisão?** (Fases 4 a 6) — baselines, modelos com sazonalidade e promoção,
  e uma política de pedido no nível de serviço de 93,75%.
